In [ ]:
import requests
import json

# Set your QuickBooks Online credentials and endpoints
ACCESS_TOKEN = "eyJlbmMiOiJBMTI4Q0JDLUhTMjU2IiwiYWxnIjoiZGlyIn0..NN_21BG6BtS9yxOVoUNUUg.Mu_XAKdHP1by1SMxHq7m-KdVZUXVHzwDy74n0EJuwOXw-cYTo538o0pIhKi-2mKFYXLsuNI1WNHSGCgSV6gGvEBMGr02UyD8Kyud3hffBFQUY9mtXbYa3VtqH4wk3z3bhQ8xapetLoe80s3tbg3Y25GMnJeSnstc99aTvuEkilVh7Wb27cdiZDX2ISDTSEMM9uyphs6jwFHsE4eEENCacz1UC1jPmANdzBou6EOpk9N_wtuu7puiJXUS5ksXvnpoysOoXgo9snp_4qyzycgSj0ccqesiC20rrYEYi4dGyke0ugOFnehyjsn0GKjE9m_trOyvhQT5ZzjQGoyPlbsx6UcNO6bcBeqYa9Q1SkcbMTvUvO0agcyV-mwilm_lgEwgywKME6H6o3wwNSOKqhtJ-aMRcWZkK1T1fYCEKKWwOivBB6kyk2nsyGp0J_b04OCEbsWzR8_YjanMcTtRFIUhEdq-issBCtyOCdBjT178ppko2wap-ocMvYvhyAKVLeU4TkEqA79odCST1uUOdA7p9ZMNP4ugelghJh4fMbzuJbvxAS91EU_2X21fGErEFiEgKnkMS1tkSbvV7bMzYvkxo5Ow5iV-FqOtEqnV9_OxC-xGgpgIdPmUAmr7pb0IsKiZvj8i0eRpwurU9NcnqEriB8Ez7TCTz0v6KbYrdGEdToA3DuheLuDBkvAdEcftdO6j9vU4Y_CNgj29JxTNrn-tFP-YLjDn-ORTP1Vq1WLt13A.uDkGIc5542UyPap0yDfuKg"       # Replace with your OAuth 2.0 access token
COMPANY_ID = "9341454212369641"           # Replace with your QuickBooks Company ID
BASE_URL = f"https://quickbooks.api.intuit.com/v3/company/{COMPANY_ID}"


: 

In [ ]:

# Setup HTTP headers for the API requests
headers = {
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "Content-Type": "application/json",
    "Accept": "application/json"
}


In [ ]:

def fetch_transactions(query="SELECT * FROM Purchase"):
    """
    Fetch transactions from QuickBooks Online using the provided query.
    Adjust the SQL-like query to select the type of transactions you need.
    """
    url = f"{BASE_URL}/query?query={query}"
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        data = response.json()
        # The example assumes we're working with Purchase transactions.
        transactions = data.get("QueryResponse", {}).get("Purchase", [])
        return transactions
    else:
        print("Error fetching transactions:", response.text)
        return []

def categorize_transaction(transaction_id, sync_token, new_account_ref, new_class_ref):
    """
    Update (categorize) a transaction by updating its line items with new account and class references.
    
    :param transaction_id: The ID of the transaction to update.
    :param sync_token: The current sync token for the transaction.
    :param new_account_ref: A dict with keys 'value' and 'name' for the expense account.
    :param new_class_ref: A dict with keys 'value' and 'name' for the class.
    """
    # Endpoint URL for updating a Purchase transaction; using the 'operation=update' parameter.
    url = f"{BASE_URL}/purchase?operation=update"
    
    # Construct the update payload.
    # Note: In a full update, you may need to include more of the original transaction data.
    payload = {
        "Purchase": {
            "Id": transaction_id,
            "SyncToken": sync_token,
            "Line": [
                {
                    "DetailType": "AccountBasedExpenseLineDetail",
                    # The amount here should match the original transaction amount or be updated as needed.
                    "Amount": 100.00,
                    "AccountBasedExpenseLineDetail": {
                        "AccountRef": new_account_ref,
                        "ClassRef": new_class_ref
                    },
                    "Description": "Updated categorization"
                }
            ]
        }
    }
    
    response = requests.post(url, headers=headers, data=json.dumps(payload))
    if response.status_code in [200, 201]:
        print(f"Transaction {transaction_id} updated successfully.")
        return response.json()
    else:
        print(f"Error updating transaction {transaction_id}: {response.text}")
        return None



In [ ]:
# Fetch transactions (this example fetches Purchase transactions)
transactions = fetch_transactions()


In [ ]:


if transactions:
    print("Fetched transactions:")
    for txn in transactions:
        print(f"ID: {txn['Id']}, TotalAmt: {txn.get('TotalAmt')}, SyncToken: {txn.get('SyncToken')}")
    
    # For demonstration, update the first transaction from the fetched list.
    first_txn = transactions[0]
    txn_id = first_txn["Id"]
    sync_token = first_txn["SyncToken"]
    
    # Define new categorization details (replace with your actual Account and Class IDs and names):
    new_account = {"value": "67", "name": "Office Supplies"}
    new_class = {"value": "89", "name": "Marketing"}
    
    update_result = categorize_transaction(txn_id, sync_token, new_account, new_class)
    if update_result:
        print("Update result:", json.dumps(update_result, indent=2))
else:
    print("No transactions found.")

In [ ]:
{
    "tools": ["list of tools"], # like this "tools": [{"tool_name": "tool name", "operation": "operations", "description": "description of the tool"}]
    "workflow": "the workflow in explanation of how to achieve the user request by using the tools for subsequesnt agent",
    "follow_ups": "ay additional follow up to ask the user before proceeding with the execution, like additional details in the parameter according to the tool that is needed to be specified by the user",   
}